In [6]:
import os 
os.environ['KERAS_BACKEND']= 'torch'
import warnings
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 
from scipy import  stats
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings('ignore')
pd.set_option('display.max_column',20)
pd.set_option('display.float_format',lambda x:'%4f' % x)

# Set global visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['figure.titlesize'] = 14

# Set reproducible random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Environment setup complete. Reproducible random seed set to 42.")



Environment setup complete. Reproducible random seed set to 42.


In [10]:
# 2.1 Ingest the dataset with fallback path support
dataset_paths=[
    'compPersonal/html/HTML_Top13_Features.csv',
    'HTML_Top13_Features.csv',
    r'D:\UUM\ASSIGNMENT\SEM5\NN\STINK3014_A261_Assignment01\compPersonal\html\HTML_Top13_Features.csv'
]

csv_path=next((p for p in dataset_paths if os.path.exists(p)),dataset_paths[0])
print(f"loading dataset from :{csv_path}")

df_raw =pd.read_csv(csv_path)

raw_cols_sample = df_raw.columns[:15].tolist()
print(f"Raw column header sample:{[repr(c) for c in raw_cols_sample]}")

df=df_raw.copy()
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include = ['object','string']).columns:
    df[col]=df[col].astype(str).str.strip()

print(f"\nSanitized column headers: {df.columns.tolist()}")
print(f"Dataset Shape: {df.shape[0]:,} rows (records) and {df.shape[1]} columns (attributes)")
print("\nFirst 5 rows of the sanitized dataset:")
df.head()



loading dataset from :D:\UUM\ASSIGNMENT\SEM5\NN\STINK3014_A261_Assignment01\compPersonal\html\HTML_Top13_Features.csv
Raw column header sample:["'file_name'", "'url_punct_char_count'", "'tag_count'", "'whitespace_ratio'", "'entropy'", "'form_count'", "'embedded_js_count'", "'html_whitespace_ratio'", "'script_entropy'", "'min_link_length'", "'external_link_count'", "'total_script_characters'", "'internal_link_count'", "'url_digit_count'", "'label'"]

Sanitized column headers: ['file_name', 'url_punct_char_count', 'tag_count', 'whitespace_ratio', 'entropy', 'form_count', 'embedded_js_count', 'html_whitespace_ratio', 'script_entropy', 'min_link_length', 'external_link_count', 'total_script_characters', 'internal_link_count', 'url_digit_count', 'label']
Dataset Shape: 19,997 rows (records) and 15 columns (attributes)

First 5 rows of the sanitized dataset:


,file_name,url_punct_char_count,tag_count,whitespace_ratio,entropy,form_count,embedded_js_count,html_whitespace_ratio,script_entropy,min_link_length,external_link_count,total_script_characters,internal_link_count,url_digit_count,label
0,sample_02317.html,28,9,0.100000,5.486996,0,0,0.033445,0.000000,62,0,1,2,148,1
1,sample_05528.html,708,125,0.250377,5.153461,1,5,0.012820,4.923656,6,80,68152,2,1049,0
2,sample_06003.html,376,114,0.278652,5.489581,1,3,0.080328,5.629287,1,33,2575,4,174,0
3,sample_07420.html,1018,354,0.422888,5.465670,2,19,0.099333,5.444942,1,43,25341,56,1706,0
4,sample_09413.html,39,652,0.572907,5.964094,0,165,0.015315,2.828016,6,1,533,13,17,1


In [ ]:
# 2.2 Data Integrity Check: Missing Values, Duplicates, and Data Types
print("=== 1. DATA TYPES & NON-NULL SUMMARY ===")
print(df.info())

print("\n=== 2.Missing Values audit====")
missing_counts=df.isnull().sum()
missing_pct = (missing_counts / len(df)*100)
missing_df=pd.DataFrame({'Missing_Count': missing_counts,'Percentage (%)':missing_pct})
print(missing_df[missing_df['Missing_Count']>0] if missing_counts.sum()>0 else 'Zero mssing values detected accross all 13 attributes')

print("\==3 Duplicate record audit===")
duplicate_rows = df.duplicated().sum()
print(f"Total duplicate records:"{duplicate_rows})

print("\n ===4 SUMARRY STATISTIC OF NUMERICAL FEATURES===")
df.describe().T[['count','mean','std','min','25%']]

=== 1. DATA TYPES & NON-NULL SUMMARY ===
<class 'pandas.DataFrame'>
RangeIndex: 19997 entries, 0 to 19996
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   file_name                19997 non-null  str    
 1   url_punct_char_count     19997 non-null  int64  
 2   tag_count                19997 non-null  int64  
 3   whitespace_ratio         19997 non-null  float64
 4   entropy                  19997 non-null  float64
 5   form_count               19997 non-null  int64  
 6   embedded_js_count        19997 non-null  int64  
 7   html_whitespace_ratio    19997 non-null  float64
 8   script_entropy           19997 non-null  float64
 9   min_link_length          19997 non-null  int64  
 10  external_link_count      19997 non-null  int64  
 11  total_script_characters  19997 non-null  int64  
 12  internal_link_count      19997 non-null  int64  
 13  url_digit_count          19997 non-null  int64